# Cyrillic HTR — transformer synthetic pretraining + augmented fine-tuning

Notebook with Colab setup, data preparation, synthetic pretraining, real fine-tuning, plots, and final held-out test evaluation.

The repository source code uses Hydra configs. Notebook constants below are Colab-only experiment overrides and paths for reproducibility.

Final evaluation uses the unified `scripts/evaluate.py`, which supports both `crnn_ctc` and `transformer_htr`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

These constants are notebook-level Hydra overrides for the Colab experiment.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/elizaveta112/cyrillic-htr.git"
REPO_DIR = Path("/content/cyrillic-htr")

DRIVE_ROOT = Path("/content/drive/MyDrive/cyrillic_htr")
EXPERIMENT_ROOT = DRIVE_ROOT / "transformer_synthetic_augmented_rescue"

SYNTH_ROOT = EXPERIMENT_ROOT / "synthetic_pretrain"
REAL_ROOT = EXPERIMENT_ROOT / "real_finetune"
FINAL_ROOT = EXPERIMENT_ROOT / "final_test"
FINAL_MODEL_ROOT = EXPERIMENT_ROOT / "final_model"

MLFLOW_DIR = DRIVE_ROOT / "mlruns"

for path in [
    DRIVE_ROOT,
    EXPERIMENT_ROOT,
    SYNTH_ROOT,
    REAL_ROOT,
    FINAL_ROOT,
    FINAL_MODEL_ROOT,
    MLFLOW_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

# Rescue preprocessing settings used as Hydra overrides below.
IMAGE_HEIGHT = 64
MAX_WIDTH = 256
IMAGE_MEAN = 0.0
IMAGE_STD = 1.0
MAX_DECODING_LENGTH = 60

# Training settings used as Hydra overrides below. Lower batch size if Colab GPU memory is tight.
SYNTHETIC_PRETRAIN_EPOCHS = 20
REAL_TARGET_EPOCHS = 60
BATCH_SIZE = 32
NUM_WORKERS = 4
SYNTHETIC_LR = 1e-4
REAL_LR = 5e-5
WEIGHT_DECAY = 1e-4

print("Experiment root:", EXPERIMENT_ROOT)
print("Synthetic pretrain root:", SYNTH_ROOT)
print("Real fine-tune root:", REAL_ROOT)
print("Final model root:", FINAL_MODEL_ROOT)
print("MLflow dir:", MLFLOW_DIR)
print("Image settings:", IMAGE_HEIGHT, MAX_WIDTH, IMAGE_MEAN, IMAGE_STD)
print("Batch size / workers:", BATCH_SIZE, NUM_WORKERS)

In [ ]:
%cd /content
!rm -rf cyrillic-htr
!git clone {REPO_URL}
%cd /content/cyrillic-htr
!git log --oneline -5

In [ ]:
!pip install -q uv
!uv sync --all-groups

In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

uv run python - <<'PY'
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
PY

## Restore real data from Kaggle


In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

mkdir -p ~/.kaggle

if [ -f "/content/drive/MyDrive/kaggle/kaggle.json" ]; then
  cp /content/drive/MyDrive/kaggle/kaggle.json ~/.kaggle/kaggle.json
  chmod 600 ~/.kaggle/kaggle.json
  echo "kaggle.json copied from Google Drive"
else
  echo "kaggle.json not found at /content/drive/MyDrive/kaggle/kaggle.json"
  echo "Create this file in Google Drive first."
  exit 1
fi

uv run kaggle datasets list -s "cyrillic handwriting" | head -20

In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

echo "Cleaning local data workspace..."
rm -rf data/raw/cyrillic-handwriting-dataset
rm -rf data/processed

echo "Downloading Kaggle dataset..."
uv run python scripts/download_data.py data.force_download=true

echo "Preparing train/val/test splits..."
uv run python scripts/prepare_splits.py

echo "Preprocessing data and building vocabulary..."
uv run python scripts/preprocess_data.py

echo "Checking processed data..."
du -sh data/processed
ls -lh data/processed

In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

uv run python - <<'PY'
import json
from pathlib import Path
import pandas as pd

vocab_path = Path("data/processed/vocab.json")
vocab = json.loads(vocab_path.read_text(encoding="utf-8"))
allowed = set(vocab)

print("vocab size:", len(vocab))
if len(vocab) != 108:
    raise RuntimeError(f"Expected vocab size 108, got {len(vocab)}")

for split_name in ["train_split", "val_split", "test_split"]:
    path = Path(f"data/processed/{split_name}.tsv")
    df = pd.read_csv(path, sep="\t")
    unknown = set()
    for text in df["text"].dropna().astype(str):
        unknown.update(ch for ch in text if ch not in allowed)
    print(split_name, "rows:", len(df), "unknown chars:", sorted(unknown))
    if unknown:
        raise RuntimeError(f"Unknown chars in {split_name}: {sorted(unknown)}")
PY

## Configure MLflow

In [ ]:
MLFLOW_TRACKING_URI = "file:///content/drive/MyDrive/cyrillic_htr/mlruns"
MLFLOW_EXPERIMENT_NAME = "cyrillic-htr-transformer-synthetic-augmented"

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME

print("MLflow tracking URI:", MLFLOW_TRACKING_URI)
print("MLflow experiment:", MLFLOW_EXPERIMENT_NAME)

## Generate small synthetic dataset

Synthetic data is used only for pretraining. Real validation and real test remain untouched for final evaluation.

In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

FONT_DIR="/content/cyrillic_htr_fonts"
mkdir -p "$FONT_DIR"

# Handwritten-looking Cyrillic-capable fonts from Google Fonts. If downloads fail,
# the generator falls back to system fonts.
wget -q -O "$FONT_DIR/MarckScript-Regular.ttf" \
  https://raw.githubusercontent.com/google/fonts/main/ofl/marckscript/MarckScript-Regular.ttf || true
wget -q -O "$FONT_DIR/BadScript-Regular.ttf" \
  https://raw.githubusercontent.com/google/fonts/main/ofl/badscript/BadScript-Regular.ttf || true
wget -q -O "$FONT_DIR/Neucha-Regular.ttf" \
  https://raw.githubusercontent.com/google/fonts/main/ofl/neucha/Neucha-Regular.ttf || true
wget -q -O "$FONT_DIR/Caveat-Regular.ttf" \
  https://raw.githubusercontent.com/google/fonts/main/ofl/caveat/Caveat%5Bwght%5D.ttf || true
wget -q -O "$FONT_DIR/Pangolin-Regular.ttf" \
  https://raw.githubusercontent.com/google/fonts/main/ofl/pangolin/Pangolin-Regular.ttf || true

uv run python - <<'PY'
from pathlib import Path
import random
import shutil

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

SYNTHETIC_TRAIN_SAMPLES = 12000
SYNTHETIC_VAL_SAMPLES = 1000

repo = Path("/content/cyrillic-htr")
out_root = repo / "data" / "synthetic_pretrain"
image_dir = out_root / "images"

if out_root.exists():
    shutil.rmtree(out_root)
image_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(repo / "data/processed/train_split.tsv", sep="\t")
texts = train_df["text"].dropna().astype(str).tolist()
texts = [text for text in texts if 1 <= len(text) <= 42]

font_paths = []
font_paths.extend(Path("/content/cyrillic_htr_fonts").glob("*.ttf"))
font_paths.extend(Path("/usr/share/fonts").rglob("*.ttf"))
font_paths = [path for path in font_paths if path.exists() and path.stat().st_size > 0]

if not font_paths:
    raise RuntimeError("No TTF fonts found for synthetic generation.")

print("Number of candidate fonts:", len(font_paths))
print("Example fonts:", [str(path) for path in font_paths[:5]])

def load_font(font_size: int) -> ImageFont.FreeTypeFont:
    for _ in range(20):
        path = random.choice(font_paths)
        try:
            return ImageFont.truetype(str(path), font_size)
        except Exception:
            continue
    return ImageFont.load_default()

def add_background_noise(image: Image.Image) -> Image.Image:
    array = np.asarray(image, dtype=np.float32)
    noise = np.random.normal(loc=0.0, scale=random.uniform(1.0, 7.0), size=array.shape)
    array = np.clip(array + noise, 0, 255)
    return Image.fromarray(array.astype(np.uint8), mode="L")

def render_text(text: str, output_path: Path) -> None:
    font_size = random.randint(24, 38)
    font = load_font(font_size)

    probe = Image.new("L", (16, 16), 255)
    draw = ImageDraw.Draw(probe)
    bbox = draw.textbbox((0, 0), text, font=font)
    text_w = max(16, bbox[2] - bbox[0])
    text_h = max(16, bbox[3] - bbox[1])

    width = min(256, max(140, text_w + random.randint(20, 70)))
    height = random.randint(58, 78)
    background = random.randint(238, 255)
    image = Image.new("L", (width, height), background)
    draw = ImageDraw.Draw(image)

    x = random.randint(6, max(7, width - text_w - 6))
    y = max(0, (height - text_h) // 2 + random.randint(-7, 7))
    ink = random.randint(0, 80)
    draw.text((x, y), text, font=font, fill=ink)

    if random.random() < 0.65:
        image = ImageEnhance.Contrast(image).enhance(random.uniform(0.65, 1.35))

    if random.random() < 0.55:
        angle = random.uniform(-5, 5)
        image = image.rotate(angle, resample=Image.Resampling.BILINEAR, expand=True, fillcolor=255)

    if random.random() < 0.55:
        shear = random.uniform(-0.12, 0.12)
        width, height = image.size
        x_shift = abs(shear) * height
        new_width = width + int(round(x_shift))
        x_offset = -x_shift if shear > 0 else 0
        image = image.transform(
            (new_width, height),
            Image.Transform.AFFINE,
            (1, shear, x_offset, 0, 1, 0),
            resample=Image.Resampling.BILINEAR,
            fillcolor=255,
        )

    if random.random() < 0.25:
        image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.15, 0.7)))

    if random.random() < 0.75:
        image = add_background_noise(image)

    image.save(output_path)

def generate_split(split_name: str, n_samples: int) -> pd.DataFrame:
    rows = []
    for idx in tqdm(range(n_samples), desc=f"generate {split_name}"):
        text = random.choice(texts)
        filename = f"{split_name}_{idx:06d}.png"
        render_text(text, image_dir / filename)
        rows.append({"image": filename, "text": text})
    return pd.DataFrame(rows)

train_synth = generate_split("synthetic_train", SYNTHETIC_TRAIN_SAMPLES)
val_synth = generate_split("synthetic_val", SYNTHETIC_VAL_SAMPLES)

train_synth.to_csv(out_root / "synthetic_train_split.tsv", sep="\t", index=False)
val_synth.to_csv(out_root / "synthetic_val_split.tsv", sep="\t", index=False)

print("Synthetic root:", out_root)
print("Synthetic images:", len(list(image_dir.glob("*.png"))))
print("Train split:", out_root / "synthetic_train_split.tsv", len(train_synth))
print("Val split:", out_root / "synthetic_val_split.tsv", len(val_synth))
PY

## Synthetic pretraining — 20 epochs

This stage trains only on synthetic data. Test is disabled.

In [ ]:
import shutil
from pathlib import Path

SYNTH_LOCAL_DIR = Path("/content/cyrillic-htr/data/synthetic_pretrain")
SYNTH_DRIVE_DIR = Path("/content/drive/MyDrive/cyrillic_htr/synthetic_pretrain")

if not SYNTH_LOCAL_DIR.exists():
    raise FileNotFoundError(SYNTH_LOCAL_DIR)

if SYNTH_DRIVE_DIR.exists():
    shutil.rmtree(SYNTH_DRIVE_DIR)

shutil.copytree(SYNTH_LOCAL_DIR, SYNTH_DRIVE_DIR)

print("Saved synthetic dataset to:", SYNTH_DRIVE_DIR)
print("Files:")
for path in sorted(SYNTH_DRIVE_DIR.iterdir()):
    print(" -", path)

In [ ]:
from pathlib import Path

import pandas as pd

SYNTH_LOCAL_DIR = Path("/content/cyrillic-htr/data/synthetic_pretrain")

train_tsv = SYNTH_LOCAL_DIR / "synthetic_train_split.tsv"
val_tsv = SYNTH_LOCAL_DIR / "synthetic_val_split.tsv"
images_dir = SYNTH_LOCAL_DIR / "images"

train_df = pd.read_csv(train_tsv, sep="\t")
val_df = pd.read_csv(val_tsv, sep="\t")

print("train rows:", len(train_df))
print("val rows:", len(val_df))
print("images:", len(list(images_dir.glob("*"))))

display(train_df.head())
display(val_df.head())

filename_col = "filename" if "filename" in train_df.columns else train_df.columns[0]

missing = []
for filename in train_df[filename_col].head(100).tolist() + val_df[filename_col].head(100).tolist():
    image_path = images_dir / Path(filename).name
    if not image_path.exists():
        missing.append(str(image_path))

print("missing sample files:", missing[:10])
if missing:
    raise RuntimeError(f"Missing image files, examples: {missing[:10]}")

In [ ]:
import numpy as np


def resolve_image_path(value):
    path = Path(str(value))
    if path.is_absolute():
        return path
    candidate = SYNTH_DIR / path
    if candidate.exists():
        return candidate
    return IMAGES_DIR / path.name


def image_ink_stats(image_path):
    img = Image.open(image_path).convert("L")
    arr = np.asarray(img).astype(np.float32)

    # Для чёрного/серого текста на светлом фоне
    dark_fraction_220 = float((arr < 220).mean())
    dark_fraction_200 = float((arr < 200).mean())
    dark_fraction_180 = float((arr < 180).mean())
    std = float(arr.std())
    mean = float(arr.mean())

    return {
        "mean": mean,
        "std": std,
        "dark_fraction_220": dark_fraction_220,
        "dark_fraction_200": dark_fraction_200,
        "dark_fraction_180": dark_fraction_180,
    }


def find_suspicious(df, max_rows=None):
    rows = []
    subset = df if max_rows is None else df.head(max_rows)

    for idx, row in subset.iterrows():
        image_path = resolve_image_path(row[filename_col])

        if not image_path.exists():
            rows.append(
                {
                    "idx": idx,
                    "image_path": str(image_path),
                    "text": row[text_col],
                    "reason": "missing_file",
                    "dark_fraction_220": 0,
                    "dark_fraction_200": 0,
                    "dark_fraction_180": 0,
                    "std": 0,
                    "mean": 255,
                }
            )
            continue

        stats = image_ink_stats(image_path)

        # Порог можно потом подстроить.
        # Если тёмных пикселей почти нет, картинка, вероятно, пустая.
        is_blank_like = stats["dark_fraction_220"] < 0.001 and stats["dark_fraction_200"] < 0.0005

        if is_blank_like:
            rows.append(
                {
                    "idx": idx,
                    "image_path": str(image_path),
                    "text": row[text_col],
                    "reason": "blank_like",
                    **stats,
                }
            )

    return pd.DataFrame(rows)


susp_train = find_suspicious(train_df)
susp_val = find_suspicious(val_df)

print("Suspicious train:", len(susp_train), "of", len(train_df))
print("Suspicious val:", len(susp_val), "of", len(val_df))

display(susp_train.head(20))
display(susp_val.head(20))

In [ ]:
import shutil
from pathlib import Path

SYNTH_DIR = Path("/content/cyrillic-htr/data/synthetic_pretrain")


def clean_split(df, suspicious_df, original_tsv, cleaned_tsv):
    bad_indices = set(suspicious_df["idx"].tolist()) if len(suspicious_df) else set()

    clean_df = df.drop(index=list(bad_indices)).reset_index(drop=True)

    backup_tsv = original_tsv.with_suffix(".backup.tsv")
    if not backup_tsv.exists():
        shutil.copy2(original_tsv, backup_tsv)

    clean_df.to_csv(cleaned_tsv, sep="\t", index=False)

    print("Original:", original_tsv)
    print("Backup:", backup_tsv)
    print("Cleaned:", cleaned_tsv)
    print("Before:", len(df), "After:", len(clean_df), "Removed:", len(df) - len(clean_df))

    return clean_df


clean_train_tsv = SYNTH_DIR / "synthetic_train_split_clean.tsv"
clean_val_tsv = SYNTH_DIR / "synthetic_val_split_clean.tsv"

clean_train_df = clean_split(train_df, susp_train, train_tsv, clean_train_tsv)
clean_val_df = clean_split(val_df, susp_val, val_tsv, clean_val_tsv)

In [ ]:
import shutil
from pathlib import Path

SYNTH_LOCAL_DIR = Path("/content/cyrillic-htr/data/synthetic_pretrain")
SYNTH_DRIVE_DIR = Path("/content/drive/MyDrive/cyrillic_htr/synthetic_pretrain")

if SYNTH_DRIVE_DIR.exists():
    shutil.rmtree(SYNTH_DRIVE_DIR)

shutil.copytree(SYNTH_LOCAL_DIR, SYNTH_DRIVE_DIR)

print("Saved cleaned synthetic dataset to:", SYNTH_DRIVE_DIR)

In [ ]:
from pathlib import Path

import pandas as pd

SYNTH_DIR = Path("/content/cyrillic-htr/data/synthetic_pretrain")

for name in [
    "synthetic_train_split.tsv",
    "synthetic_val_split.tsv",
    "synthetic_train_split_clean.tsv",
    "synthetic_val_split_clean.tsv",
    "synthetic_train_split.backup.tsv",
    "synthetic_val_split.backup.tsv",
]:
    path = SYNTH_DIR / name
    print(name, "exists:", path.exists())
    if path.exists() and path.suffix == ".tsv":
        df = pd.read_csv(path, sep="\t")
        print("  rows:", len(df))

In [ ]:
import shutil
from pathlib import Path

import pandas as pd

SYNTH_DIR = Path("/content/cyrillic-htr/data/synthetic_pretrain")

shutil.copy2(
    SYNTH_DIR / "synthetic_train_split_clean.tsv",
    SYNTH_DIR / "synthetic_train_split.tsv",
)

shutil.copy2(
    SYNTH_DIR / "synthetic_val_split_clean.tsv",
    SYNTH_DIR / "synthetic_val_split.tsv",
)

train_df_check = pd.read_csv(SYNTH_DIR / "synthetic_train_split.tsv", sep="\t")
val_df_check = pd.read_csv(SYNTH_DIR / "synthetic_val_split.tsv", sep="\t")

print("train rows:", len(train_df_check))
print("val rows:", len(val_df_check))
print("images:", len(list((SYNTH_DIR / "images").glob("*"))))

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

SYNTH_LOCAL_DIR = Path("/content/cyrillic-htr/data/synthetic_pretrain")
PREVIEW_PATH = Path("/content/drive/MyDrive/cyrillic_htr/synthetic_pretrain_preview.png")

train_df = pd.read_csv(SYNTH_LOCAL_DIR / "synthetic_train_split_clean.tsv", sep="\t")
images_dir = SYNTH_LOCAL_DIR / "images"

filename_col = "filename" if "filename" in train_df.columns else train_df.columns[0]
text_col = "text" if "text" in train_df.columns else train_df.columns[-1]

sample = train_df.sample(min(16, len(train_df)), random_state=42)

fig, axes = plt.subplots(4, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample.iterrows(), strict=False):
    image_path = images_dir / Path(row[filename_col]).name
    image = Image.open(image_path).convert("RGB")
    ax.imshow(image, cmap="gray")
    ax.set_title(str(row[text_col])[:50], fontsize=9)
    ax.axis("off")

for ax in axes[len(sample) :]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(PREVIEW_PATH, dpi=150)
plt.show()

print("Saved preview:", PREVIEW_PATH)

In [ ]:
%%bash
set -e

SRC="/content/cyrillic-htr/data/synthetic_pretrain/"
DST="/content/drive/MyDrive/cyrillic_htr/synthetic_pretrain/"

rm -rf "$DST"
mkdir -p "$DST"

rsync -ah --info=progress2 "$SRC" "$DST"

echo "Copied synthetic_pretrain to Drive:"
du -sh "$DST"
find "$DST" -maxdepth 2 -type f | wc -l

In [ ]:
from pathlib import Path

import pandas as pd

SYNTH_DRIVE_DIR = Path("/content/drive/MyDrive/cyrillic_htr/synthetic_pretrain")

train_tsv = SYNTH_DRIVE_DIR / "synthetic_train_split.tsv"
val_tsv = SYNTH_DRIVE_DIR / "synthetic_val_split.tsv"
images_dir = SYNTH_DRIVE_DIR / "images"

print("Drive dir exists:", SYNTH_DRIVE_DIR.exists())
print("images exists:", images_dir.exists())
print("train tsv exists:", train_tsv.exists())
print("val tsv exists:", val_tsv.exists())

train_df = pd.read_csv(train_tsv, sep="\t")
val_df = pd.read_csv(val_tsv, sep="\t")

print("train rows:", len(train_df))
print("val rows:", len(val_df))
print("image files:", len(list(images_dir.glob("*"))))

In [ ]:
from pathlib import Path

import pandas as pd

SYNTH_DRIVE_DIR = Path("/content/drive/MyDrive/cyrillic_htr/synthetic_pretrain")
IMAGES_DIR = SYNTH_DRIVE_DIR / "images"

train_df = pd.read_csv(SYNTH_DRIVE_DIR / "synthetic_train_split.tsv", sep="\t")
val_df = pd.read_csv(SYNTH_DRIVE_DIR / "synthetic_val_split.tsv", sep="\t")


def get_filename_col(df):
    for col in ["filename", "image", "path", "image_path"]:
        if col in df.columns:
            return col
    return df.columns[0]


filename_col = get_filename_col(train_df)
print("filename_col:", filename_col)


def resolve_image_path(value):  # noqa: F811
    path = Path(str(value))
    if path.is_absolute():
        return path
    candidate = SYNTH_DRIVE_DIR / path
    if candidate.exists():
        return candidate
    return IMAGES_DIR / path.name


def check_missing(df, split_name):
    missing = []
    for value in df[filename_col].tolist():
        image_path = resolve_image_path(value)
        if not image_path.exists():
            missing.append(str(image_path))

    print(f"{split_name}: rows={len(df)}, missing={len(missing)}")
    if missing:
        print("Examples:", missing[:10])
    return missing


missing_train = check_missing(train_df, "train")
missing_val = check_missing(val_df, "val")

assert len(missing_train) == 0
assert len(missing_val) == 0

print("All TSV-referenced images exist on Drive.")

In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

SYNTHETIC_PRETRAIN_EPOCHS=20
BATCH_SIZE=32
SYNTHETIC_LR=0.0001

export MPLBACKEND=Agg
export HYDRA_FULL_ERROR=1
export MLFLOW_TRACKING_URI="file:///content/drive/MyDrive/cyrillic_htr/mlruns"
export MLFLOW_EXPERIMENT_NAME="cyrillic-htr-transformer-synthetic-pretrain-rescue"

DRIVE_SYN_ROOT="/content/drive/MyDrive/cyrillic_htr/transformer_synthetic_augmented_rescue/synthetic_pretrain"
LOCAL_SYN_ROOT="/content/synthetic_pretrain_run"

LOCAL_MODEL_DIR="$LOCAL_SYN_ROOT/models"
LOCAL_LOG_DIR="$LOCAL_SYN_ROOT/logs"
LOCAL_PLOTS_DIR="$LOCAL_SYN_ROOT/plots"
LOCAL_HYDRA_DIR="$LOCAL_SYN_ROOT/hydra"

DRIVE_MODEL_DIR="$DRIVE_SYN_ROOT/models"
DRIVE_LOG_DIR="$DRIVE_SYN_ROOT/logs"
DRIVE_PLOTS_DIR="$DRIVE_SYN_ROOT/plots"
DRIVE_HYDRA_DIR="$DRIVE_SYN_ROOT/hydra"

rm -rf "$LOCAL_SYN_ROOT"
mkdir -p "$LOCAL_MODEL_DIR" "$LOCAL_LOG_DIR" "$LOCAL_PLOTS_DIR" "$LOCAL_HYDRA_DIR"
mkdir -p "$DRIVE_SYN_ROOT"

GPU_LOG="$DRIVE_SYN_ROOT/gpu_monitor_synthetic_pretrain_cleaned.csv"

nvidia-smi \
  --query-gpu=timestamp,name,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw \
  --format=csv \
  -l 60 > "$GPU_LOG" &

MONITOR_PID=$!
trap "kill $MONITOR_PID 2>/dev/null || true" EXIT

uv run python scripts/train.py \
  model=transformer_htr \
  train=colab_transformer \
  data.dataset_dir=data/synthetic_pretrain/images \
  data.train_split_tsv=data/synthetic_pretrain/synthetic_train_split.tsv \
  data.val_split_tsv=data/synthetic_pretrain/synthetic_val_split.tsv \
  data.test_split_tsv=data/synthetic_pretrain/synthetic_val_split.tsv \
  data.augmentation.enabled=false \
  data.max_width=256 \
  data.image_mean=0.0 \
  data.image_std=1.0 \
  model.max_decoding_length=60 \
  train.best_metric=val_cer \
  train.save_top_k=3 \
  train.epochs="$SYNTHETIC_PRETRAIN_EPOCHS" \
  train.batch_size="$BATCH_SIZE" \
  train.num_workers=4 \
  train.learning_rate="$SYNTHETIC_LR" \
  train.limit_test_batches=0 \
  train.save_dir="$LOCAL_MODEL_DIR" \
  train.log_dir="$LOCAL_LOG_DIR" \
  train.plots_dir="$LOCAL_PLOTS_DIR" \
  logger.enabled=true \
  logger.tracking_uri="$MLFLOW_TRACKING_URI" \
  logger.experiment_name="$MLFLOW_EXPERIMENT_NAME" \
  dvc.enabled=false \
  hydra.run.dir="$LOCAL_HYDRA_DIR"

kill "$MONITOR_PID" 2>/dev/null || true

mkdir -p "$DRIVE_MODEL_DIR" "$DRIVE_LOG_DIR" "$DRIVE_PLOTS_DIR" "$DRIVE_HYDRA_DIR"

rsync -ah --delete "$LOCAL_MODEL_DIR/" "$DRIVE_MODEL_DIR/"
rsync -ah --delete "$LOCAL_LOG_DIR/" "$DRIVE_LOG_DIR/"
rsync -ah --delete "$LOCAL_PLOTS_DIR/" "$DRIVE_PLOTS_DIR/"
rsync -ah --delete "$LOCAL_HYDRA_DIR/" "$DRIVE_HYDRA_DIR/"

cp "$DRIVE_MODEL_DIR/last.ckpt" "$DRIVE_MODEL_DIR/checkpoint_20.ckpt"

echo "Synthetic pretraining checkpoint:"
ls -lh "$DRIVE_MODEL_DIR"

echo "Synthetic logs:"
find "$DRIVE_LOG_DIR" -maxdepth 3 -type f | head -20

echo "Synthetic plots:"
ls -lh "$DRIVE_PLOTS_DIR"

echo "Last GPU monitor records:"
tail -20 "$GPU_LOG"

## Real fine-tuning with train augmentations

This stage initializes model weights from the synthetic pretraining checkpoint, then trains on real train data with augmentation. Real validation is used for checkpoint selection. Real test is disabled during staged training.

In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

export MPLBACKEND=Agg
export HYDRA_FULL_ERROR=1
export MLFLOW_TRACKING_URI="file:///content/drive/MyDrive/cyrillic_htr/mlruns"
export MLFLOW_EXPERIMENT_NAME="cyrillic-htr-transformer-real-augmented-rescue"

# Change manually: 5, 10, 20, 30, 50, 80, 100, 128
export TARGET_EPOCHS=60

SYN_MODEL_DIR="/content/drive/MyDrive/cyrillic_htr/transformer_synthetic_augmented_rescue/synthetic_pretrain/models"

REAL_ROOT="/content/drive/MyDrive/cyrillic_htr/transformer_synthetic_augmented_rescue/real_finetune"
MODEL_DIR="$REAL_ROOT/models"
LOG_DIR="$REAL_ROOT/logs"
PLOTS_DIR="$REAL_ROOT/plots"
HYDRA_DIR="$REAL_ROOT/hydra"

mkdir -p "$MODEL_DIR" "$LOG_DIR" "$PLOTS_DIR" "$HYDRA_DIR"

EXTRA_ARGS=()

if [ -f "$MODEL_DIR/last.ckpt" ]; then
  COMPLETED_EPOCHS=$(uv run python - <<PY
import torch
checkpoint = torch.load("$MODEL_DIR/last.ckpt", map_location="cpu", weights_only=False)
print(int(checkpoint.get("epoch", -1)) + 1)
PY
)

  echo "Found real fine-tuning checkpoint: $MODEL_DIR/last.ckpt"
  echo "Completed real fine-tuning epochs: $COMPLETED_EPOCHS"

  if [ "$COMPLETED_EPOCHS" -ge "$TARGET_EPOCHS" ]; then
    echo "Checkpoint already has $COMPLETED_EPOCHS completed epochs, but TARGET_EPOCHS=$TARGET_EPOCHS."
    echo "Increase TARGET_EPOCHS."
    exit 1
  fi

  EXTRA_ARGS+=("train.resume_from_checkpoint=$MODEL_DIR/last.ckpt")
else
  SYN_CKPT=$(ls -t "$SYN_MODEL_DIR"/best-*.ckpt 2>/dev/null | head -1 || true)

  if [ -z "$SYN_CKPT" ]; then
    SYN_CKPT="$SYN_MODEL_DIR/last.ckpt"
  fi

  if [ ! -f "$SYN_CKPT" ]; then
    echo "No synthetic checkpoint found. Run synthetic pretraining cell first."
    exit 1
  fi

  SAFE_SYN_CKPT="$SYN_MODEL_DIR/synthetic_best.ckpt"
  cp "$SYN_CKPT" "$SAFE_SYN_CKPT"

  echo "No real fine-tuning checkpoint found."
  echo "Original synthetic checkpoint: $SYN_CKPT"
  echo "Safe synthetic checkpoint: $SAFE_SYN_CKPT"
  echo "Initializing weights from synthetic checkpoint."

  EXTRA_ARGS+=("train.init_from_checkpoint=$SAFE_SYN_CKPT")
fi

GPU_LOG="$REAL_ROOT/gpu_monitor_target_${TARGET_EPOCHS}.csv"

nvidia-smi \
  --query-gpu=timestamp,name,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw \
  --format=csv \
  -l 60 > "$GPU_LOG" &

MONITOR_PID=$!
trap "kill $MONITOR_PID 2>/dev/null || true" EXIT

uv run python scripts/train.py \
  model=transformer_htr \
  train=colab_transformer \
  data.augmentation.enabled=true \
  data.max_width=256 \
  data.image_mean=0.0 \
  data.image_std=1.0 \
  model.max_decoding_length=60 \
  train.best_metric=val_cer \
  train.save_top_k=1 \
  train.epochs="$TARGET_EPOCHS" \
  train.batch_size=32 \
  train.num_workers=4 \
  train.learning_rate=0.00005 \
  train.limit_test_batches=0 \
  train.save_dir="$MODEL_DIR" \
  train.log_dir="$LOG_DIR" \
  train.plots_dir="$PLOTS_DIR" \
  logger.enabled=true \
  logger.tracking_uri="$MLFLOW_TRACKING_URI" \
  logger.experiment_name="$MLFLOW_EXPERIMENT_NAME" \
  dvc.enabled=false \
  hydra.run.dir="$HYDRA_DIR" \
  "${EXTRA_ARGS[@]}"

kill "$MONITOR_PID" 2>/dev/null || true

cp "$MODEL_DIR/last.ckpt" "$MODEL_DIR/checkpoint_${TARGET_EPOCHS}.ckpt"
echo "Saved real fine-tuning checkpoint: $MODEL_DIR/checkpoint_${TARGET_EPOCHS}.ckpt"

echo "Last GPU monitor records:"
tail -20 "$GPU_LOG"

### Early stopping



In [ ]:
from pathlib import Path

import pandas as pd

LOG_DIR = Path(
    "/content/drive/MyDrive/cyrillic_htr/transformer_synthetic_augmented_rescue/real_finetune/logs"
)

metrics_files = sorted(LOG_DIR.rglob("metrics.csv"))
df = pd.concat([pd.read_csv(p) for p in metrics_files], ignore_index=True)

val_rows = df[df["val_cer"].notna()].copy()
val_rows = val_rows.sort_values("epoch")

cols = [
    "epoch",
    "step",
    "val_loss",
    "val_cer",
    "val_wer",
    "val_line_accuracy",
    "val_edit_similarity",
]
cols = [c for c in cols if c in val_rows.columns]

print("Best by val_cer:")
display(val_rows.sort_values("val_cer")[cols].head(10))

print("Last 20 validation points:")
display(val_rows[cols].tail(20))

best = val_rows.loc[val_rows["val_cer"].idxmin()]
last = val_rows.iloc[-1]

print("Best epoch:", int(best["epoch"]))
print("Best val_cer:", best["val_cer"])
print("Last epoch:", int(last["epoch"]))
print("Last val_cer:", last["val_cer"])

recent = val_rows[val_rows["epoch"] >= val_rows["epoch"].max() - 10]
recent_best = recent["val_cer"].min()
older_best = val_rows[val_rows["epoch"] < val_rows["epoch"].max() - 10]["val_cer"].min()

print("Best before last 10 epochs:", older_best)
print("Best in last 10 epochs:", recent_best)
print("Improvement in last 10 epochs:", older_best - recent_best)

## Inspect checkpoints and metrics

In [ ]:
%%bash
set -e

cd /content/cyrillic-htr

uv run python - <<'PY'
from pathlib import Path
import torch

for name, model_dir in {
    "synthetic_pretrain": Path("/content/drive/MyDrive/cyrillic_htr/transformer_synthetic_augmented_rescue/synthetic_pretrain/models"),
    "real_finetune": Path("/content/drive/MyDrive/cyrillic_htr/transformer_synthetic_augmented_rescue/real_finetune/models"),
}.items():
    print(f"\n{name}:")
    for path in sorted(model_dir.glob("*.ckpt")):
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"  {path.name}: {size_mb:.2f} MB")
    last_ckpt = model_dir / "last.ckpt"
    if last_ckpt.exists():
        checkpoint = torch.load(last_ckpt, map_location="cpu", weights_only=False)
        print("  completed epochs:", int(checkpoint.get("epoch", -1)) + 1)
PY

In [ ]:
from pathlib import Path

import pandas as pd

log_root = Path(
    "/content/drive/MyDrive/cyrillic_htr/transformer_synthetic_augmented_rescue/real_finetune/logs"
)
metrics_files = sorted(log_root.glob("csv/version_*/metrics.csv"))

if not metrics_files:
    raise FileNotFoundError(f"No metrics.csv found under {log_root}")

print("Metrics files:")
for path in metrics_files:
    print(" -", path)

metrics = pd.read_csv(metrics_files[-1])
display(metrics.tail(30))

if "val_cer" in metrics.columns:
    val_rows = metrics.dropna(subset=["val_cer"])
    if not val_rows.empty:
        display(val_rows.loc[val_rows["val_cer"].idxmin()])